# ARC Tritium Fuel Cycle

The residence time model of the tritium fuel cycle of an ARC-class fusion power plant, from Meschini et al., *Nucl. Fusion* **63**, 126005 (2023), built from the fuel cycle component blocks of `pathsim_fusion`. The plasma is pulsed, and the example follows the tritium inventories through the first ten days of operation.

Every component holds one tritium inventory $I$. It is filled by its inflows and leaves towards the next component after the residence time $\tau$. A fraction $\varepsilon$ of the outflow is lost on top of it, and the inventory decays with the constant $\lambda$:

$$\frac{dI}{dt} = \sum_i u_i - (1 + \varepsilon) \frac{I}{\tau} - \lambda I$$

The model has two loops that meet in the storage:

- **Outer fuel cycle.** Tritium bred in the breeding zone goes to the tritium extraction system. The extracted part passes a membrane on its way to the storage. The rest reaches the heat exchanger and returns to the breeding zone, the first wall and the divertor, except for a small part that permeates and goes to the detritiation system.
- **Inner fuel cycle.** The unburnt fuel is pumped out of the fusion chamber. A part is recycled directly to the storage, the rest passes the fuel cleanup and the isotope separation. The isotope separation sends its waste stream to the detritiation system, which returns the recovered tritium.

In [ ]:
import matplotlib.pyplot as plt

plt.style.use("../pathsim_docs.mplstyle")

from pathsim import Connection, Simulation
from pathsim.blocks import Adder, Amplifier, Integrator, PulseSource, Scope, Splitter
from pathsim.solvers import RKBS32

from pathsim_fusion import (
    BreedingBlanket,
    DetritiationSystem,
    FuelCycleComponent,
    FusionChamberPump,
    HeatExchangerPermeation,
    IsotopeSeparationSystem,
    PlasmaFacingComponent,
    StorageAndManagement,
    TokamakExhaustProcessing,
    TritiumExtractionSystem,
)

## Parameters

Inventories are in kg and times in seconds. The values are those of the model published with the paper (github.com/SamueleMeschini/fuel-cycle). Three of them differ from the tables of the paper: the residence times of the fuel cleanup (1 h, in the paper 0.3 h) and of the isotope separation (3 h, in the paper 3.7 h), and the tritium burn rate (8.99e-7 kg/s, in the paper 9.3e-7 kg/s).

In [ ]:
HOUR = 3600.0
DAY = 24 * HOUR

# residence times [s]
tau_bz = 1.25 * HOUR  # breeding zone
tau_tes = 24 * HOUR  # tritium extraction system
tau_fw = 1000.0  # first wall
tau_div = 1000.0  # divertor
tau_hx = 1000.0  # heat exchanger
tau_ds = 1 * HOUR  # detritiation system
tau_vp = 600.0  # vacuum pump
tau_fc = 1 * HOUR  # fuel cleanup
tau_iss = 3 * HOUR  # isotope separation
tau_mem = 100.0  # membrane

decay_constant = 1.73e-9  # tritium decay constant [1/s]
loss_fraction = 1e-4  # non-radioactive loss fraction [-]

# flow fractions [-]
f_p_fw = 1e-4  # plasma exhaust to first wall
f_p_div = 1e-4  # plasma exhaust to divertor
f_hx_bz = 0.33  # heat exchanger to breeding zone
f_hx_fw = 0.33  # heat exchanger to first wall
f_hx_ds = 1e-4  # heat exchanger to detritiation
f_hx_div = 1 - f_hx_bz - f_hx_fw - f_hx_ds  # heat exchanger to divertor
f_iss_ds = 0.1  # isotope separation to detritiation
f_dir = 0.3  # direct internal recycling, vacuum pump to storage
eta_tes = 0.95  # extraction efficiency

# plasma
burn_rate = 8.99e-7  # tritium burn rate [kg/s]
tbe = 0.02  # tritium burn efficiency [-]
tbr = 1.067  # tritium breeding ratio [-]
pulse_period = 1800.0  # [s]
availability = 0.7  # fraction of each period with plasma [-]

startup_inventory = 1.14  # tritium in the storage at the start [kg]

## Components

Each component of the fuel cycle has a block. The blocks with two outlets split their outflow by a fraction: the extraction efficiency, the permeated fraction, the directly recycled fraction and the waste fraction. The membrane has no block of its own and uses the generic `FuelCycleComponent`. In the reference model the first wall and the divertor have no non-radioactive loss.

In [ ]:
common = {"loss_fraction": loss_fraction, "decay_constant": decay_constant}

bz = BreedingBlanket(tau=tau_bz, **common)
tes = TritiumExtractionSystem(tau=tau_tes, efficiency=eta_tes, **common)
fw = PlasmaFacingComponent(tau=tau_fw, decay_constant=decay_constant)
div = PlasmaFacingComponent(tau=tau_div, decay_constant=decay_constant)
hx = HeatExchangerPermeation(tau=tau_hx, permeation_fraction=f_hx_ds, **common)
ds = DetritiationSystem(tau=tau_ds, **common)
vp = FusionChamberPump(tau=tau_vp, dir_fraction=f_dir, **common)
fc = TokamakExhaustProcessing(tau=tau_fc, **common)
iss = IsotopeSeparationSystem(tau=tau_iss, waste_fraction=f_iss_ds, **common)
mem = FuelCycleComponent(tau=tau_mem, **common)
sto = StorageAndManagement(
    decay_constant=decay_constant, initial_value=startup_inventory
)

names = ["BZ", "TES", "FW", "DIV", "HX", "DS", "VP", "FC", "ISS", "MEM", "STO"]
components = [bz, tes, fw, div, hx, ds, vp, fc, iss, mem, sto]

## Plasma

The plasma holds no inventory in this model. It is a pulse that is on for 70 % of each 30 minute period, scaled to the tritium flows it drives. With the burn rate $\dot{N}$ and the burn efficiency $TBE$, the fuelling takes $\dot{N} / TBE$ from the storage. The burnt tritium breeds $TBR \, \dot{N}$ in the breeding zone, and the unburnt fuel goes to the vacuum pump, apart from the small fractions that reach the first wall and the divertor.

In [ ]:
pulse = PulseSource(amplitude=1.0, T=pulse_period, duty=availability)

fuelling = Amplifier(-burn_rate / tbe)  # withdrawn from the storage
breeding = Amplifier(tbr * burn_rate)
to_fw = Amplifier(f_p_fw * burn_rate / tbe)
to_div = Amplifier(f_p_div * burn_rate / tbe)
to_vp = Amplifier((1 - tbe - f_p_fw - f_p_div) * burn_rate / tbe)

# what returns from the heat exchanger goes to the blanket, first wall and divertor
returned = 1 - f_hx_ds
split_hx = Splitter([f_hx_bz / returned, f_hx_fw / returned, f_hx_div / returned])

## Connections

All inventories are recorded once per minute. The `loss` outputs of the components are summed and integrated, which gives the tritium that left the cycle by decay and non-radioactive losses.

In [ ]:
sco = Scope(sampling_period=60.0, labels=[*names, "lost"])

loss_rate = Adder()
lost = Integrator(0.0)

connections = [
    Connection(pulse, fuelling, breeding, to_fw, to_div, to_vp),
    # plasma and breeding
    Connection(breeding, bz[0]),
    Connection(to_fw, fw[0]),
    Connection(to_div, div[0]),
    Connection(to_vp, vp[0]),
    Connection(fuelling, sto[0]),
    # outer fuel cycle
    Connection(bz["outflow"], tes[0]),
    Connection(tes["extracted"], mem[0]),
    Connection(tes["residual"], hx[0]),
    Connection(mem["outflow"], sto[1]),
    Connection(hx["returned"], split_hx),
    Connection(split_hx[0], bz[1]),
    Connection(split_hx[1], fw[1]),
    Connection(split_hx[2], div[1]),
    Connection(hx["permeated"], ds[0]),
    Connection(fw["outflow"], bz[2]),
    Connection(div["outflow"], bz[3]),
    # inner fuel cycle
    Connection(vp["recycled"], sto[2]),
    Connection(vp["exhaust"], fc[0]),
    Connection(fc["outflow"], iss[0]),
    Connection(ds["outflow"], iss[1]),
    Connection(iss["product"], sto[3]),
    Connection(iss["waste"], ds[1]),
    # tritium that left the cycle
    Connection(loss_rate, lost),
    Connection(lost, sco[len(components)]),
]
for i, component in enumerate(components):
    connections.append(Connection(component["inventory"], sco[i]))
    connections.append(Connection(component["loss"], loss_rate[i]))

blocks = [
    *components,
    pulse,
    fuelling,
    breeding,
    to_fw,
    to_div,
    to_vp,
    split_hx,
    loss_rate,
    lost,
    sco,
]

sim = Simulation(
    blocks,
    connections,
    Solver=RKBS32,
    dt=1.0,
    dt_max=pulse_period,
    tolerance_lte_rel=1e-6,
    tolerance_lte_abs=1e-12,
    log=True,
)

sim.run(10 * DAY)

## Inventories

The inventories of the components build up from empty. The isotope separation holds by far the most tritium outside the storage. The extraction system, with its residence time of one day, takes the longest to fill.

In [ ]:
time, data = sco.read()
days = time / DAY
inventory = dict(zip(names, data))

labels = {
    "ISS": "isotope separation",
    "TES": "tritium extraction",
    "FC": "fuel cleanup",
    "VP": "vacuum pump",
    "BZ": "breeding zone",
}

fig, ax = plt.subplots()
for name, label in labels.items():
    ax.plot(days, inventory[name], label=label)
ax.set_yscale("log")
ax.set_ylim(1e-4, 1)
ax.set_xlabel("time [days]")
ax.set_ylabel("tritium inventory [kg]")
ax.legend(ncol=2)
plt.show()

The storage empties while the other components fill up, reaches its lowest inventory after about three days, and then grows with the net breeding. The lowest point shows how much of the startup inventory the cycle needs.

In [ ]:
i_sto = inventory["STO"]

fig, ax = plt.subplots()
ax.plot(days, i_sto)
ax.set_xlabel("time [days]")
ax.set_ylabel("storage inventory [kg]")
plt.show()

print(
    f"lowest storage inventory: {i_sto.min():.3f} kg after {days[i_sto.argmin()]:.1f} days"
)

The pulses of the plasma are visible in the components with a short residence time. The vacuum pump fills while the plasma is on and empties in between, the breeding zone follows more slowly.

In [ ]:
hours = time / HOUR
window = (hours >= 48) & (hours <= 51)

fig, ax = plt.subplots()
ax.plot(hours[window], inventory["VP"][window] * 1e3, label="vacuum pump")
ax.plot(hours[window], inventory["BZ"][window] * 1e3, label="breeding zone")
ax.set_xlabel("time [hours]")
ax.set_ylabel("tritium inventory [g]")
ax.set_ylim(0, 30)
ax.legend(ncol=2, loc="upper right")
plt.show()

## Tritium Balance

As a check, all inventories plus the tritium that was lost must equal the startup inventory plus the net breeding, $(TBR - 1) \, \dot{N}$ times the time the plasma was on:

In [ ]:
total = data.sum(axis=0)  # all inventories and the integrated losses

on_time = pulse_period * availability
burn_time = (time // pulse_period) * on_time + (time % pulse_period).clip(max=on_time)
expected = startup_inventory + (tbr - 1) * burn_rate * burn_time

print(f"largest deviation from the balance: {abs(total - expected).max():.1e} kg")